In [ ]:
import re
import subprocess

from pathlib import Path


LLM_MODEL = ""
TIMEOUT_SECONDS = 600

GAME = "nine_mens_morris"
OPEN_SPIEL_MAX_STEPS = 500

OUTPUTS_DIR = Path("code/outputs")
RESPONSE_PATH = OUTPUTS_DIR / f"{GAME}.md"
CODE_PATH = OUTPUTS_DIR / f"{GAME}.py"


## LLM implementation

This cell reads `code/input/prompt.txt` and `code/input/game_rules.txt` directly.


In [ ]:
try:
    if not LLM_MODEL:
        raise ValueError("Set LLM_MODEL")

    prompt_text = Path("code/input/prompt.txt").read_text(encoding="utf-8")
    rules_text = Path("code/input/game_rules.txt").read_text(encoding="utf-8")
    full_prompt = prompt_text + "\n\nHier folgt die Spielanleitung:\n\n" + rules_text

    result = subprocess.run(
        ["pi", "-p", "--model", LLM_MODEL, full_prompt],
        capture_output=True,
        text=True,
        timeout=TIMEOUT_SECONDS,
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip() or "pi call failed")

    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    RESPONSE_PATH.write_text(result.stdout, encoding="utf-8")

    match = re.search(r"```python\s*(.*?)```", result.stdout, re.IGNORECASE | re.DOTALL)
    if match is None:
        raise RuntimeError("No fenced python block found in the LLM response")

    CODE_PATH.write_text(match.group(1).strip() + "\n", encoding="utf-8")
except Exception as exc:
    print(f"LLM call failed: {exc}")


## OpenSpiel and generated game loading


In [ ]:
import importlib.util
import pyspiel

try:
    game = pyspiel.load_game(GAME)

    if not CODE_PATH.exists():
        raise FileNotFoundError(f"Generated code missing: {CODE_PATH}")

    spec = importlib.util.spec_from_file_location("generated_game", CODE_PATH)
    module = importlib.util.module_from_spec(spec)
    if spec is None or spec.loader is None:
        raise RuntimeError("Could not load generated game module")
    spec.loader.exec_module(module)
    llm_game = module.Game()
except Exception as exc:
    print(f"Game loading failed: {exc}")


## Random test on both games


In [ ]:
import random

try:
    state = game.new_initial_state()
    llm_state = llm_game.initial_state()
    steps = 0

    while not state.is_terminal():
        action = random.choice(state.legal_actions())
        state.apply_action(action)

        llm_actions = list(llm_game.legal_actions(llm_state))
        if action not in llm_actions:
            raise RuntimeError(f"Action not legal in generated game at step {steps}: {action}")

        next_state = llm_game.apply_action(llm_state, action)
        if next_state is not None:
            llm_state = next_state

        steps += 1
        if steps > OPEN_SPIEL_MAX_STEPS:
            raise RuntimeError(f"Too many OpenSpiel steps: {OPEN_SPIEL_MAX_STEPS}")
except NameError:
    print("Random test failed: load the games first")
except Exception as exc:
    print(f"Random test failed: {exc}")
